# 140 — Audio Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/140-audio-agent/audio_agent_workbook.ipynb)

**What you'll learn:**
- How OpenAI Whisper converts audio to text in a single API call
- How to design a JSON-schema-in-prompt for structured intent classification
- How a three-node LangGraph pipeline routes support tickets end-to-end
- Why routing should be a separate, token-free lookup node
- How to handle fence-stripping, fallback routes, and edge cases gracefully
- How to extend the routing table and confidence thresholds without touching core logic

**Source:** `examples/140-audio-agent/src/`

**Requirements:** `openai`, `python-dotenv`, `langgraph`

| Concept | Why it matters |
|---------|----------------|
| Whisper `audio.transcriptions.create` | Converts speech to text — supports 50+ languages, one API call |
| Intent classification prompt | JSON-schema-in-prompt forces a typed, enumerable intent field |
| `ROUTING_MAP` lookup | Maps intent strings to queue names — extend without changing logic |
| Fence-stripping | `strip("\`\`\`json")` before `json.loads()` handles LLM markdown wrapping |
| Fail-fast / graceful skip | Part B checks for the audio file and skips cleanly if absent |
| Confidence filtering | Route below threshold to human review regardless of intent |

In [ ]:
%pip install -q openai python-dotenv langgraph

### Quick Reference — Costs and Limits

| Resource | Limit / Cost |
|----------|-------------|
| Whisper API file size | 25 MB max per request |
| Whisper API pricing | $0.006 / minute of audio |
| GPT-4o-mini input | $0.15 / 1M tokens |
| GPT-4o-mini output | $0.60 / 1M tokens |
| Classification prompt size | ~150 tokens (system + schema) |
| Typical analysis response | ~80–120 tokens |
| Cost per call (Whisper + classify) | ~$0.006 + ~$0.00004 ≈ **$0.006/call** |

At 1,000 support calls/day: ~**$6/day** or ~**$180/month**.
At 10,000 calls/day: ~$60/day. At that scale, evaluate self-hosted Whisper + fine-tuned classifier.

> **Rate limits (Tier 1 defaults):** Whisper: 500 req/min. GPT-4o-mini: 500 req/min, 200k TPM.
> For batch processing, use the [Batch API](https://platform.openai.com/docs/guides/batch) at 50% discount.

---
## Part A — Concepts and Simulation

Cells A1–A12 are pure Python. No API key required. You will:
- Understand Whisper model tradeoffs
- Learn audio preprocessing requirements
- Trace intent classification design decisions
- Simulate the full pipeline with mock transcripts
- Complete two hands-on exercises

### A1 — Whisper Model Comparison

OpenAI's Whisper comes in several sizes. The hosted API always uses `whisper-1` (equivalent to the `large-v2` checkpoint), but understanding the full family helps when evaluating self-hosted alternatives.

| Model | WER (English) | Speed (RTF) | VRAM | Best use case |
|-------|--------------|-------------|------|---------------|
| tiny | ~14% | ~32x | 1 GB | Edge devices, offline, latency-critical demos |
| base | ~11% | ~16x | 1 GB | Local development, fast iteration |
| small | ~9% | ~6x | 2 GB | Balanced accuracy/speed on consumer GPU |
| medium | ~7% | ~2x | 5 GB | Production on a single GPU, good multilingual |
| large-v3 | ~5% | ~1x (baseline) | 10 GB | Best accuracy, server-side batch processing |
| whisper-1 (API) | ~5% | n/a (hosted) | n/a | Default choice — no infra required |

**RTF** (Real-Time Factor): a value of 32x means the model processes 32 seconds of audio per second of wall-clock time.  
**WER** (Word Error Rate): lower is better; values approximate English Common Voice benchmarks.

> For this example we use `whisper-1` via the OpenAI API. If you want to self-host, start with `base` for development and `large-v3` for production.

### A2 — Audio Preprocessing Requirements

Whisper is opinionated about input. Getting this wrong is the #1 source of poor transcription quality.

| Parameter | Whisper requirement | Why |
|-----------|--------------------|----- |
| Sample rate | 16 kHz (resampled internally if higher) | Model was trained at 16 kHz; resampling at source is safer |
| Channels | Mono preferred (stereo accepted) | Stereo doubles file size with no quality gain for STT |
| Formats | mp3, mp4, mpeg, mpga, m4a, wav, webm, ogg | No flac, aiff, or opus via API |
| Max file size | 25 MB (API limit) | Split long recordings into segments before upload |
| Duration sweet spot | 30s – 5 min per segment | Very short clips (<5s) increase relative error rate |

**Quick conversion with ffmpeg:**
```bash
# Record on macOS and convert to 16kHz mono WAV
say -o /tmp/sample.aiff "Hi, I was charged twice this month and need a refund."
ffmpeg -i /tmp/sample.aiff -ar 16000 -ac 1 /tmp/sample.wav
```

**Noise matters:** Whisper handles background noise well but improves significantly with noise-cancelled input. Run a simple high-pass filter (cutoff ~80 Hz) to remove low-frequency rumble before sending to the API.

### A3 — Intent Classification Approaches

There are three common approaches to classifying user intent from text. Choose based on your accuracy, latency, and maintenance budget.

| Approach | Accuracy | Latency | Maintenance cost | When to use |
|----------|----------|---------|-----------------|-------------|
| Regex / keyword rules | Low–medium | <1 ms | High (brittle, grows fast) | Toy projects, very narrow domains |
| Embedding similarity | Medium | ~10–50 ms | Medium (need labelled examples) | Stable intent set, cost-sensitive |
| LLM prompt | High | ~200–800 ms | Low (update prompt, not code) | Broad domain, handles paraphrase naturally |

**This example uses the LLM prompt approach** because:
1. The intent set is small and well-defined (5 categories)
2. Support calls use wildly varied phrasing — embeddings need hundreds of labelled examples
3. We can enforce a JSON schema directly in the prompt without a separate validation step

**Trade-off to know:** LLM classification costs ~0.5–2 cents per call at GPT-4o-mini pricing. At 10,000 calls/day that is $50–200/day. At that scale, consider fine-tuning a small classifier or switching to embedding similarity for the high-volume route.

### A4 — The Transcribe → Classify → Route Pipeline

```
audio file
    │
    ▼
┌─────────────────────┐
│  transcribe_node    │  Whisper API → plain text transcript
│  (I/O bound)        │  ~200-800 ms depending on audio length
└─────────────────────┘
    │  state["transcript"] = "Hi, I was charged twice..."
    ▼
┌─────────────────────┐
│  analyze_node       │  GPT-4o-mini → JSON: intent, urgency,
│  (LLM call)         │  sentiment, product_mentioned, action_required
└─────────────────────┘
    │  state["analysis"] = {"intent": "billing", "urgency": "medium", ...}
    ▼
┌─────────────────────┐
│  route_node         │  ROUTING_MAP dict lookup → queue name
│  (pure Python)      │  0 tokens, 0 ms latency
└─────────────────────┘
    │  state["queue"] = "billing-queue"
    ▼
downstream handler
```

**State object** shared across all three nodes:
```python
class AudioState(TypedDict):
    audio_path: str   # input: path to the audio file
    transcript: str   # populated by transcribe_node
    analysis: dict    # populated by analyze_node
    queue: str        # populated by route_node
```

**Why keep routing separate from analysis?**  
The `route_node` is a pure dictionary lookup — zero tokens, zero API latency. Separating it means you can swap the routing table without touching the classification prompt, and you can test the router without mocking the LLM.

In [ ]:
# A5 — Routing table with example transcripts for all 5 intent categories

ROUTING_MAP = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "cancellation": "retention-team",
    "general": "general-support",
}

MOCK_TRANSCRIPTS = {
    "billing": (
        "Hi, I received my invoice this month and it looks like I was charged twice "
        "for my subscription. The amount is 89 dollars and I only signed up once. "
        "Can you please look into this and issue a refund?"
    ),
    "technical": (
        "Hello, I'm having trouble logging into the dashboard. Every time I enter "
        "my credentials it gives me an error 403 message. I've tried resetting my "
        "password twice but the problem persists."
    ),
    "complaint": (
        "I am very disappointed with the service I received last week. The agent "
        "I spoke to was rude and my issue was never resolved. I want to file a "
        "formal complaint and speak to a manager."
    ),
    "cancellation": (
        "I would like to cancel my account. I've been a customer for two years but "
        "the pricing has gone up too much and I found a cheaper alternative. "
        "Please process the cancellation before the next billing cycle."
    ),
    "general": (
        "Hi, I just wanted to know what your business hours are and whether "
        "you offer live chat support. I have a quick question about setting "
        "up my profile."
    ),
}


def route(intent: str) -> str:
    """Map intent string to queue name. Unknown intents fall back to general-support."""
    return ROUTING_MAP.get(intent, "general-support")


print("Intent routing table:")
print("=" * 50)
for intent, queue in ROUTING_MAP.items():
    print(f"  {intent:15} → {queue}")
print()
print("Example transcripts loaded for 5 intents.")
print(f"Fallback for unknown intent: '{route('unknown')}'")

In [ ]:
# A6 — Build the classification prompt with JSON schema
# The prompt instructs the LLM to return structured JSON — no post-processing needed

SYSTEM_PROMPT = """You are a support call classifier. Analyze the transcript and respond with JSON only.
Use exactly this schema:
{
  "intent": "billing|technical|general|complaint|cancellation",
  "confidence": <float 0.0-1.0>,
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative",
  "entities": ["list", "of", "key", "entities"]
}
Respond with JSON only. No markdown fences. No explanation."""


def build_prompt(transcript: str) -> str:
    return f"{SYSTEM_PROMPT}\n\nTranscript: {transcript}"


# Print the prompt for the billing example
sample_prompt = build_prompt(MOCK_TRANSCRIPTS["billing"])
print("Classification prompt (billing example):")
print("-" * 60)
print(sample_prompt)
print("-" * 60)
print()
print("Key design choices:")
print("  - Pipe-separated enums force the model to pick exactly one value")
print("  - confidence (0.0-1.0) enables downstream threshold filtering")
print("  - entities list catches names, product IDs, amounts for CRM enrichment")
print("  - 'No markdown fences' reduces but doesn't eliminate fence wrapping")

In [ ]:
# A7 — Simulate routing: given mock LLM responses, show which queue fires

import json

MOCK_LLM_RESPONSES = {
    "billing": '{"intent": "billing", "confidence": 0.96, "urgency": "medium", "product_mentioned": "subscription", "action_required": "Verify charge history and issue refund for duplicate payment", "sentiment": "negative", "entities": ["invoice", "89 dollars", "subscription"]}',
    "technical": '```json\n{"intent": "technical", "confidence": 0.91, "urgency": "high", "product_mentioned": "dashboard", "action_required": "Investigate 403 error and reset session token", "sentiment": "neutral", "entities": ["403", "dashboard", "credentials"]}\n```',
    "complaint": '  {"intent": "complaint", "confidence": 0.88, "urgency": "high", "product_mentioned": null, "action_required": "Escalate to manager and log formal complaint", "sentiment": "negative", "entities": ["manager", "formal complaint"]}  ',
    "cancellation": '{"intent": "cancellation", "confidence": 0.94, "urgency": "medium", "product_mentioned": null, "action_required": "Offer retention discount before processing cancellation", "sentiment": "negative", "entities": ["account", "billing cycle", "two years"]}',
    "general": '{"intent": "general", "confidence": 0.82, "urgency": "low", "product_mentioned": null, "action_required": "Send FAQ link for business hours and live chat info", "sentiment": "positive", "entities": ["business hours", "live chat", "profile"]}',
}


def strip_fences(raw: str) -> dict:
    """Strip markdown fences and leading/trailing whitespace before parsing JSON."""
    text = raw.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


print("Full pipeline simulation (mock transcripts, no API call):")
print("=" * 65)
for intent_key, transcript in MOCK_TRANSCRIPTS.items():
    analysis = strip_fences(MOCK_LLM_RESPONSES[intent_key])
    detected_intent = analysis.get("intent", "general")
    confidence = analysis.get("confidence", 0.0)
    queue = route(detected_intent)
    print(f"\nTranscript : {transcript[:70]}...")
    print(f"Intent     : {detected_intent} (confidence: {confidence:.0%})")
    print(f"Urgency    : {analysis.get('urgency')}")
    print(f"Queue      : {queue}")
print("\n" + "=" * 65)

### A7b — Why Fence-Stripping Is Necessary

The `strip_fences()` helper handles a very common LLM quirk: even when you say
"respond with JSON only", many models wrap the output in markdown code fences.

**What the model returns vs. what we need:**
```
What we asked for:   {"intent": "billing", ...}
What we often get:   ```json
                     {"intent": "billing", ...}
                     ```
```

**The three-step strip:**
```python
text = raw.strip()           # remove leading/trailing whitespace
         .strip("```json")   # remove opening fence with language hint
         .strip("```")       # remove closing fence
         .strip()            # clean up any remaining whitespace
```

**Edge cases this handles:**
- `\`\`\`json\n{...}\n\`\`\`` — standard fenced code block
- `  {"intent": ...}  ` — leading/trailing whitespace only
- `\`\`\`\n{...}\n\`\`\`` — fence without language hint

**Edge cases this does NOT handle:**
- Multi-key JSON returned as a list `[{...}, {...}]`
- Model returning prose explanation before the JSON
- Truncated JSON (model hit max_tokens mid-output)

For production, wrap `json.loads()` in a try/except and log the `{"raw": text}` fallback
to a separate error queue for human review.

In [ ]:
# A8 — Simulate 3 edge cases
# 1. Empty transcript → fallback route
# 2. Two intents in one message → highest-confidence wins
# 3. No match → "unknown" queue

print("Edge case 1: Empty transcript")
print("-" * 50)
# When Whisper returns an empty string (silence, music, noise-only audio)
empty_transcript = ""
if not empty_transcript.strip():
    fallback_result = {"intent": "general", "confidence": 0.0, "urgency": "low",
                       "action_required": "No speech detected — route to review queue"}
    queue = route(fallback_result["intent"])
    print(f"  No speech detected → intent='{fallback_result['intent']}' → queue='{queue}'")
    print(f"  Confidence: {fallback_result['confidence']} → would trigger human review")

print()
print("Edge case 2: Two intents in one message (billing + complaint)")
print("-" * 50)
# Customer says 'I was charged wrong AND the agent was rude'
# LLM returns two candidate intents — take highest confidence
multi_intent_responses = [
    {"intent": "billing", "confidence": 0.72, "urgency": "medium"},
    {"intent": "complaint", "confidence": 0.61, "urgency": "high"},
]
best = max(multi_intent_responses, key=lambda x: x["confidence"])
queue = route(best["intent"])
print(f"  Candidates: {[(r['intent'], r['confidence']) for r in multi_intent_responses]}")
print(f"  Winner     : '{best['intent']}' (conf={best['confidence']}) → queue='{queue}'")

print()
print("Edge case 3: Truly unrecognized intent")
print("-" * 50)
# LLM returns an intent string not in the routing map (hallucination or new category)
unknown_intent = "partnership_inquiry"
queue = route(unknown_intent)
print(f"  Intent '{unknown_intent}' not in ROUTING_MAP → fallback queue='{queue}'")
print("  Action: log to 'unclassified' table for periodic review and routing map updates")

### A9 — Routing Architecture Options

Two common patterns for routing in multi-intent systems:

| Approach | Central Router | Per-Intent Classifier |
|----------|---------------|----------------------|
| Structure | One LLM call returns intent + metadata | N classifiers, one per intent |
| Latency | ~1 LLM call per request | ~N LLM calls per request |
| Accuracy | Good for small intent sets (<10) | Better for large, overlapping intents |
| Maintenance | Update one prompt | Update N prompts independently |
| Explainability | Single JSON response | Per-intent confidence scores |
| This example | Yes (analyze_node) | — |

**When to use per-intent classifiers:**
- You have more than ~10 intent categories
- Intents frequently overlap (e.g., billing + complaint in same message)
- Different intents need different extraction schemas
- You need independent confidence thresholds per intent

For this support-ticket use case, the central router is the right choice: small intent set, one primary intent per call, and a single JSON structure covers all metadata we need downstream.

### A10 — Confidence Calibration

LLMs are not inherently calibrated — a model that says `"confidence": 0.9` is not necessarily right 90% of the time. The confidence field in our prompt is a **soft signal**, not a probability estimate.

**How to calibrate in practice:**

1. Collect 500+ labelled examples from your actual call audio
2. Run the pipeline and record `(predicted_intent, confidence, true_intent)`
3. Bucket by confidence range (0.0–0.5, 0.5–0.7, 0.7–0.85, 0.85–1.0)
4. For each bucket, compute accuracy = `correct / total`
5. Plot confidence vs. accuracy — a well-calibrated model has a near-diagonal curve

**Practical thresholds (starting point, tune with your data):**

| Confidence | Action |
|-----------|--------|
| ≥ 0.85 | Auto-route to queue |
| 0.65 – 0.85 | Route but flag for spot-check |
| < 0.65 | Send to `human_review` queue regardless of intent |

**GPT-4o-mini calibration note:** In practice, the model tends to output high confidence (0.8–0.95) even for borderline cases. Setting your auto-route threshold at 0.85 rather than 0.7 is a safer starting point until you have real calibration data.

In [ ]:
# A11 — Build and display a routing decision log
# In production, this becomes an audit table in your database

import time

decision_log = []

for intent_key, transcript in MOCK_TRANSCRIPTS.items():
    analysis = strip_fences(MOCK_LLM_RESPONSES[intent_key])
    detected_intent = analysis.get("intent", "general")
    confidence = analysis.get("confidence", 0.0)
    queue = route(detected_intent)
    auto_routed = confidence >= 0.85

    decision_log.append({
        "transcript_preview": transcript[:60] + "...",
        "intent": detected_intent,
        "confidence": confidence,
        "queue": queue,
        "auto_routed": auto_routed,
        "urgency": analysis.get("urgency"),
    })

print("Routing Decision Log")
print("=" * 80)
header = f"{'TRANSCRIPT':<45} {'INTENT':<14} {'CONF':>5} {'QUEUE':<18} {'AUTO?'}"
print(header)
print("-" * 80)
for entry in decision_log:
    auto = "YES" if entry["auto_routed"] else "REVIEW"
    print(f"{entry['transcript_preview']:<45} {entry['intent']:<14} {entry['confidence']:>4.0%} {entry['queue']:<18} {auto}")
print("=" * 80)
auto_count = sum(1 for e in decision_log if e["auto_routed"])
print(f"\nAuto-routed: {auto_count}/{len(decision_log)} | Review queue: {len(decision_log)-auto_count}/{len(decision_log)}")

In [ ]:
# A11b — LangGraph state pattern: why TypedDict and immutable updates
# Each node receives the full state and returns a FULL updated copy.
# This immutable pattern makes the pipeline easy to test and debug.

from typing import TypedDict

class AudioState(TypedDict):
    audio_path: str
    transcript: str
    analysis: dict
    queue: str

# Simulate how each node receives and returns state
initial_state: AudioState = {
    "audio_path": "/tmp/sample.wav",
    "transcript": "",
    "analysis": {},
    "queue": "",
}

# Node 1: transcribe_node updates transcript field
after_transcribe = {**initial_state, "transcript": "Hi, I was charged twice..."}

# Node 2: analyze_node updates analysis field
after_analyze = {**after_transcribe, "analysis": {"intent": "billing", "confidence": 0.96}}

# Node 3: route_node updates queue field
after_route = {**after_analyze, "queue": route(after_analyze["analysis"]["intent"])}

print("State evolution through the pipeline:")
print("=" * 55)
print(f"initial_state  → transcript='{initial_state['transcript'] or '(empty)'}'")
print(f"after_transcribe → transcript='{after_transcribe['transcript'][:40]}...'")
print(f"after_analyze  → intent='{after_analyze['analysis']['intent']}'")
print(f"after_route    → queue='{after_route['queue']}'")
print()
print("Key: {**state, 'field': new_value} creates a NEW dict — original is unchanged.")
print("This immutability lets you replay any node in isolation for debugging.")

### A12 — Batch vs. Streaming Transcription

Whisper via the OpenAI API is a **batch** operation: you upload a complete audio file and receive a complete transcript. For many use cases this is fine. But some pipelines need **streaming** transcription.

| Mode | Latency | Use case | How |
|------|---------|----------|-----|
| Batch (this example) | Full audio length + ~1–3s | Voicemail, recorded calls, asynchronous | `audio.transcriptions.create` |
| Streaming (chunked) | ~500ms first token | Live call assistance, real-time captions | WebSockets or chunked HTTP with self-hosted model |
| Real-time (WebRTC) | <200ms | Voice assistants, interruption detection | Deepgram Nova, AssemblyAI Streaming, or whisper.cpp with VAD |

**For this example** we use batch mode: the use case is routing inbound support voicemails, which arrive as complete audio files — not live streams.

**If you need streaming:** The architecture is the same (transcribe → classify → route) but `transcribe_node` becomes an async generator that yields transcript fragments, and `analyze_node` buffers until a sentence boundary before classifying.

> See example **102-voice-pipeline** for a live voice pipeline with real-time STT.

---
### Exercise 1 — Add a 'complaint' Intent Alias

The current routing table sends 'complaint' to `customer-success`. Your task:

1. Add a **6th intent** called `"complaint_escalation"` that routes to `"human-escalation"` (for calls with urgency=high AND negative sentiment)
2. Update `SYSTEM_PROMPT` to include `complaint_escalation` in the pipe-separated enum
3. Update the `route()` function call to handle the new intent
4. Verify the decision log shows `human-escalation` for a matching test case

**Hint:** You only need to change `ROUTING_MAP` and `SYSTEM_PROMPT`. The `route()` function and `analyze_node` require no changes.

In [ ]:
# Exercise 1: Add a 6th intent 'complaint_escalation' to the routing table
# that sends high-urgency, negative-sentiment complaints to human escalation.
# Update the classification system prompt to include it.

# TODO: Update ROUTING_MAP_EX1 to include 'complaint_escalation' → 'human-escalation'
ROUTING_MAP_EX1 = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "cancellation": "retention-team",
    "general": "general-support",
    # TODO: add 'complaint_escalation' here
}

# TODO: Update SYSTEM_PROMPT_EX1 to add 'complaint_escalation' to the intent enum
SYSTEM_PROMPT_EX1 = """You are a support call classifier. Respond with JSON only.
{
  "intent": "billing|technical|general|complaint|cancellation",  # TODO: add complaint_escalation
  "confidence": <float 0.0-1.0>,
  "urgency": "low|medium|high",
  "sentiment": "positive|neutral|negative"
}"""

# TODO: Add a mock transcript that should trigger 'complaint_escalation'
escalation_transcript = "TODO: write a transcript with high urgency and very negative sentiment"

print("Exercise 1 — TODO items:")
print("  [ ] Add 'complaint_escalation' → 'human-escalation' to ROUTING_MAP_EX1")
print("  [ ] Update SYSTEM_PROMPT_EX1 intent enum to include complaint_escalation")
print("  [ ] Write a mock transcript that should trigger the new intent")
print("  [ ] Simulate the routing and confirm 'human-escalation' queue fires")

In [ ]:
# Exercise 1 Answer Key
# Adding complaint_escalation requires only ROUTING_MAP and SYSTEM_PROMPT changes.
# The route() function, analyze_node, and transcribe_node are untouched.

ROUTING_MAP_ANSWER = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "complaint_escalation": "human-escalation",  # new: high-urgency complaints
    "cancellation": "retention-team",
    "general": "general-support",
}

SYSTEM_PROMPT_ANSWER = """You are a support call classifier. Respond with JSON only.
Use complaint_escalation (not complaint) when urgency is high AND sentiment is negative.
{
  "intent": "billing|technical|general|complaint|complaint_escalation|cancellation",
  "confidence": <float 0.0-1.0>,
  "urgency": "low|medium|high",
  "sentiment": "positive|neutral|negative",
  "entities": ["list"]
}"""

# Mock response for a high-urgency escalation case
escalation_mock_response = '{"intent": "complaint_escalation", "confidence": 0.92, "urgency": "high", "sentiment": "negative", "entities": ["manager", "legal action", "three calls"]}'

analysis_ex1 = strip_fences(escalation_mock_response)
queue_ex1 = ROUTING_MAP_ANSWER.get(analysis_ex1["intent"], "general-support")

print("Exercise 1 Answer Key:")
print("-" * 50)
print(f"Intent    : {analysis_ex1['intent']}")
print(f"Confidence: {analysis_ex1['confidence']:.0%}")
print(f"Urgency   : {analysis_ex1['urgency']}")
print(f"Queue     : {queue_ex1}")
print()
# Key insight: only ROUTING_MAP and SYSTEM_PROMPT changed — no LangGraph node changes needed
print("Key insight: the routing table is the only extensibility point.")
print("route() already handles unknown keys via .get() fallback — zero code changes needed.")

---
### Exercise 2 — Confidence Threshold Filtering

Right now, every classified intent routes directly to its queue regardless of confidence. That means a `"billing"` classification at 0.3 confidence goes straight to `billing-queue`. That is dangerous in production.

Your task: write a `route_or_escalate()` function that:
1. If `confidence >= 0.7`, routes using the normal `ROUTING_MAP`
2. If `confidence < 0.7`, routes to `"human_review"` regardless of intent
3. Returns a dict with `{"queue": ..., "auto_routed": bool, "reason": ...}`

**Test it** against these three cases:
- `{"intent": "billing", "confidence": 0.92}` → should auto-route
- `{"intent": "technical", "confidence": 0.61}` → should go to human_review
- `{"intent": "unknown_intent", "confidence": 0.88}` → unknown intent, auto-route to fallback

In [ ]:
# Exercise 2: Write a route_or_escalate() function with confidence threshold filtering.
# If confidence < 0.7, route to 'human_review' regardless of intent.

CONFIDENCE_THRESHOLD = 0.70

# TODO: implement route_or_escalate()
def route_or_escalate(analysis: dict) -> dict:
    """
    Route a classified intent to the appropriate queue.
    If confidence < CONFIDENCE_THRESHOLD, send to 'human_review' instead.
    Returns: {"queue": str, "auto_routed": bool, "reason": str}
    """
    # TODO: implement this function
    pass


# Test cases
test_cases = [
    {"intent": "billing", "confidence": 0.92, "urgency": "medium"},
    {"intent": "technical", "confidence": 0.61, "urgency": "high"},
    {"intent": "unknown_intent", "confidence": 0.88, "urgency": "low"},
]

print("Exercise 2 — Test cases (run after implementing route_or_escalate):")
for tc in test_cases:
    result = route_or_escalate(tc)
    print(f"  intent={tc['intent']:20} conf={tc['confidence']} → result={result}")

In [ ]:
# Exercise 2 Answer Key

def route_or_escalate_answer(analysis: dict, threshold: float = 0.70) -> dict:
    """
    Route a classified intent to the appropriate queue with confidence gating.
    If confidence < threshold, route to 'human_review' regardless of intent.
    """
    intent = analysis.get("intent", "general")
    confidence = analysis.get("confidence", 0.0)

    if confidence < threshold:
        return {
            "queue": "human_review",
            "auto_routed": False,
            "reason": f"confidence {confidence:.0%} below threshold {threshold:.0%}",
        }

    queue = ROUTING_MAP.get(intent, "general-support")
    return {
        "queue": queue,
        "auto_routed": True,
        "reason": f"intent='{intent}' confidence {confidence:.0%} above threshold",
    }


test_cases = [
    {"intent": "billing", "confidence": 0.92, "urgency": "medium"},
    {"intent": "technical", "confidence": 0.61, "urgency": "high"},
    {"intent": "unknown_intent", "confidence": 0.88, "urgency": "low"},
]

print("Exercise 2 Answer Key:")
print("-" * 70)
for tc in test_cases:
    result = route_or_escalate_answer(tc)
    flag = "AUTO" if result["auto_routed"] else "REVIEW"
    print(f"  [{flag:6}] intent={tc['intent']:20} conf={tc['confidence']} → queue='{result['queue']}'")
    print(f"           reason: {result['reason']}")
print()
print("Key insight: threshold filtering is a separate concern from routing.")
print("Keeping them separate means you can tune the threshold without touching ROUTING_MAP.")

---
## Part B — Live API Run

The cells below require:
1. `OPENAI_API_KEY` in your `.env` file
2. An audio file at `/tmp/sample.wav` (WAV, MP3, M4A, or any Whisper-supported format)

**Don't have a sample audio file?** Create one with macOS `say`:
```bash
say -o /tmp/sample.aiff "Hi, I was charged twice this month and I need a refund right away."
ffmpeg -i /tmp/sample.aiff -ar 16000 -ac 1 /tmp/sample.wav
```

**Cell B1** checks both prerequisites. If the audio file is missing, it runs in **DEMO MODE** with a mock transcript so you can still see the full pipeline output without an audio file.

In [ ]:
# B1 — Fail-fast setup: check OPENAI_API_KEY and audio file

import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

AUDIO_PATH = "/tmp/sample.wav"  # change to your audio file path
_READY = False
_DEMO_MODE = False

api_key = os.environ.get("OPENAI_API_KEY", "")

if not api_key:
    print("SKIP: OPENAI_API_KEY not set. Set it in .env and re-run.")
else:
    client = OpenAI(api_key=api_key)
    try:
        client.models.list()
        print("Connected to OpenAI API.")
        if Path(AUDIO_PATH).exists():
            file_size = Path(AUDIO_PATH).stat().st_size
            print(f"Audio file : {AUDIO_PATH} ({file_size:,} bytes)")
            _READY = True
        else:
            print(f"Audio file not found at '{AUDIO_PATH}'.")
            print("Running in DEMO MODE with mock transcript.")
            print("To use real audio: say -o /tmp/sample.aiff 'I need a refund'")
            print("                  ffmpeg -i /tmp/sample.aiff -ar 16000 -ac 1 /tmp/sample.wav")
            _DEMO_MODE = True
            _READY = True
    except Exception as e:
        print(f"SKIP: Cannot reach OpenAI API: {e}")

In [ ]:
# B2 — Run the full transcribe → analyze → route workflow

if not _READY:
    print("Skipping — prerequisites not met (see Cell B1 output).")
else:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict

    class AudioState(TypedDict):
        audio_path: str
        transcript: str
        analysis: dict
        queue: str

    def transcribe_node(state: AudioState, config: dict) -> AudioState:
        if _DEMO_MODE:
            mock_transcript = (
                "Hi, I received my invoice this month and I was charged twice for my subscription. "
                "The amount is 89 dollars and I only signed up once. Can you please look into this "
                "and issue a refund? I've been a customer for three years."
            )
            return {**state, "transcript": mock_transcript}
        _client = config["configurable"]["client"]
        with open(state["audio_path"], "rb") as f:
            result = _client.audio.transcriptions.create(model="whisper-1", file=f)
        return {**state, "transcript": result.text}

    def analyze_node(state: AudioState, config: dict) -> AudioState:
        _client = config["configurable"]["client"]
        model = config["configurable"].get("model", "gpt-4o-mini")
        prompt = f"""{SYSTEM_PROMPT}

Transcript: {state['transcript']}"""
        resp = _client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
        )
        raw = resp.choices[0].message.content
        analysis = strip_fences(raw)
        return {**state, "analysis": analysis}

    def route_node(state: AudioState, config: dict) -> AudioState:
        intent = state["analysis"].get("intent", "general")
        return {**state, "queue": route(intent)}

    graph = StateGraph(AudioState)
    graph.add_node("transcribe", transcribe_node)
    graph.add_node("analyze", analyze_node)
    graph.add_node("route", route_node)
    graph.set_entry_point("transcribe")
    graph.add_edge("transcribe", "analyze")
    graph.add_edge("analyze", "route")
    graph.add_edge("route", END)
    workflow = graph.compile()

    cfg = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

    if _DEMO_MODE:
        print("=" * 60)
        print("DEMO MODE (no audio file — using mock transcript)")
        print("=" * 60)
    else:
        print(f"Processing: {AUDIO_PATH}")

    result = workflow.invoke(
        {"audio_path": AUDIO_PATH, "transcript": "", "analysis": {}, "queue": ""},
        config=cfg,
    )

    print(f"\nTranscript : {result['transcript']}")
    print(f"Analysis   : {json.dumps(result['analysis'], indent=2)}")
    print(f"\nRouted to  : {result['queue']}")

In [ ]:
# B3 — Print routing summary with timing and queue reference

if not _READY:
    print("Skipping — prerequisites not met (see Cell B1 output).")
else:
    import time

    analysis = result["analysis"]
    intent = analysis.get("intent", "unknown")
    confidence = analysis.get("confidence", 0.0)
    urgency = analysis.get("urgency", "unknown")
    sentiment = analysis.get("sentiment", "unknown")
    queue = result["queue"]

    print("Routing Summary")
    print("=" * 50)
    print(f"Intent       : {intent}")
    print(f"Confidence   : {confidence:.0%}" if isinstance(confidence, float) else f"Confidence   : {confidence}")
    print(f"Urgency      : {urgency}")
    print(f"Sentiment    : {sentiment}")
    print(f"Queue        : {queue}")

    # Apply confidence-based routing from Exercise 2
    routing_decision = route_or_escalate_answer(analysis)
    print(f"Auto-routed  : {'YES' if routing_decision['auto_routed'] else 'NO — sent to human_review'}")
    print()
    print("Queue reference:")
    for i, (k, v) in enumerate(ROUTING_MAP.items()):
        marker = "<-- this call" if v == queue else ""
        print(f"  {k:15} → {v} {marker}")

In [ ]:
# B4 — Inspect the full AudioState object after workflow completion

if not _READY:
    print("Skipping — prerequisites not met (see Cell B1 output).")
else:
    print("Full AudioState after workflow completion:")
    print("=" * 60)
    for key, value in result.items():
        if not key.startswith("_"):
            label = key.upper()
            if isinstance(value, dict):
                print(f"{label}:")
                print(json.dumps(value, indent=2))
            else:
                print(f"{label}: {value}")
    print("=" * 60)
    print()
    print("Entities extracted:")
    entities = result["analysis"].get("entities", [])
    if entities:
        for e in entities:
            print(f"  - {e}")
    else:
        print("  (none returned)")
    print()
    print("Action required:")
    print(f"  {result['analysis'].get('action_required', 'n/a')}")


---
## What's Next

- **Whisper paper** — Radford et al. (2022) *Robust Speech Recognition via Large-Scale Weak Supervision*: [arxiv.org/abs/2212.04356](https://arxiv.org/abs/2212.04356) — explains the training data curation and multi-task framing that makes Whisper so robust.
- **OpenAI Audio API docs** — [platform.openai.com/docs/guides/speech-to-text](https://platform.openai.com/docs/guides/speech-to-text) — covers language detection, timestamps, word-level segmentation, and the `verbose_json` response format.
- **Real-time audio with WebSockets** — For live call routing (<500ms latency), look at [Deepgram Streaming](https://developers.deepgram.com/docs/streaming) or [AssemblyAI Real-Time](https://www.assemblyai.com/docs/speech-to-text/streaming). The LangGraph routing pattern from this example composes directly.
- **Example 138 — Vision Q&A Agent** — The same three-node pipeline pattern (load → extract → classify) applied to images instead of audio. Good next step for multimodal support ticket routing (audio + screenshot).